# Set up environment

⚠️ Must restart session after installing HMMER package in order for ANARCI to install and run correctly.

⏳ 5 minutes

In [1]:
# @title Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title Install Packages
!pip install biopython #required by ANARCI
!pip install fair-esm torch transformers #required for ESM-2 embeddings
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install tensorflow
!pip install matplotlib
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.7 MB/s eta 0:00:00


In [3]:
# @title Install HMMER
!apt-get install -y hmmer #required by ANARCI

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libdivsufsort3
Suggested packages:
  hmmer-doc
The following NEW packages will be installed:
  hmmer libdivsufsort3
0 upgraded, 2 newly installed, 0 to remove and 42 not upgraded.
Need to get 1,198 kB of archives.
After this operation, 7,621 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libdivsufsort3 amd64 2.0.1-5 [42.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 hmmer amd64 3.3.2+dfsg-1 [1,155 kB]
Fetched 1,198 kB in 1s (1,318 kB/s)
Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_am

In [ ]:
# @title Restart Session
import os
os.kill(os.getpid(), 9)

In [1]:
# @title  Clone and install ANARCI
!git clone https://github.com/oxpig/ANARCI.git
%cd ANARCI
!python setup.py install

Cloning into 'ANARCI'...
remote: Enumerating objects: 793, done.
remote: Counting objects: 100% (330/330), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 793 (delta 273), reused 227 (delta 227), pack-reused 463 (from 1)
Receiving objects: 100% (793/793), 7.11 MiB | 27.05 MiB/s, done.
Resolving deltas: 100% (454/454), done.
/content/ANARCI
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
INFO: ANARCI lives in:  /usr/local/lib/python3.12/dist

In [2]:
# @title Load Libraries

# ===== Data tools =====
import pandas as pd
import numpy as np
import csv
import os
import gc
import joblib

# ===== Bioinformatics =====
from anarci import anarci
from collections import Counter

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Deep learning tools =====
import torch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# ===== Embeddings tools =====
import esm

# ===== Preprocessing & Evaluation =====
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# ===== Models =====
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

# Sequence Alignment

Goal: Numbering of protein sequences

⏳ 1 minute

In [3]:
# @title Create a working directory on Drive
output_dir = "/content/drive/MyDrive/poster_antibodies"
!mkdir -p {output_dir}
print(f"Results will be saved to: {output_dir}")

Results will be saved to: /content/drive/MyDrive/poster_antibodies


In [4]:
# @title Clean dataset

# Load CSV file
file_path = "/content/drive/MyDrive/Downloads/Colab/antibody_data.csv"
df = pd.read_csv(file_path)

# Keep only the desired columns
final_df = df[['antibody_id', 'vh_protein_sequence', 'vl_protein_sequence', 'tm2_nanodsf_avg']]

# Total number of antibodies (rows)
total_antibodies = final_df.shape[0]

# Clean dataset
df_clean = final_df.dropna(subset=["tm2_nanodsf_avg"])
df_clean = df_clean.rename(columns={"tm2_nanodsf_avg": "temp"})
remaining_antibodies = df_clean.shape[0]

print(f"Total antibodies: {total_antibodies}")
print(f"After dropping empty Tm values: {remaining_antibodies}")
print(f"Dropped rows: {total_antibodies - remaining_antibodies}")

# Define the output directory
output_dir = "/content/drive/MyDrive/poster_antibodies"

# Save dataset to the specified directory
output_filepath = os.path.join(output_dir, "antibody_temp.csv")
df_clean.to_csv(output_filepath, index=False)
print(f"CSV file '{output_filepath}' created successfully.")
print(f"Number of rows: {len(df_clean)}")

Total antibodies: 246
After dropping empty Tm values: 208
Dropped rows: 38
CSV file '/content/drive/MyDrive/poster_antibodies/antibody_temp.csv' created successfully.
Number of rows: 208


In [5]:
# @title IMGT numbering for VH sequences

# Load CSV file
input_csv = "/content/drive/MyDrive/poster_antibodies/antibody_temp.csv"
output_dir = "/content/drive/MyDrive/poster_antibodies"
output_csv = os.path.join(output_dir, "anarci_vh.csv")  #change output file name as needed

sequences = []
with open(input_csv, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        seq_id = row.get("antibody_id")
        seq = row["vh_protein_sequence"].replace(" ", "").upper()  #remove spaces, ensure uppercase
        sequences.append((seq_id, seq))

# Run ANARCI in batch
results, assignment_details, hit_tables = anarci(sequences, scheme="imgt")

# Open output CSV
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        [
            "antibody_id",
            "chain_type",
            "species",
            "bitscore",
            "start",
            "end",
            "cdr1",
            "cdr2",
            "cdr3",
            "aligned_imgt",
        ]
    )

    # Process each sequence
    for i, (seq_id, seq) in enumerate(sequences):
        if results[i] and results[i][0]:
            for domain_idx, (numbering, start, end) in enumerate(results[i]):

                def extract_region(numbering, start_pos, end_pos):
                    return "".join(
                        aa for ((pos, _), aa) in numbering if start_pos <= pos <= end_pos and aa != "-"
                    )

                cdr1 = extract_region(numbering, 27, 38)
                cdr2 = extract_region(numbering, 56, 65)
                cdr3 = extract_region(numbering, 105, 117)
                aligned_seq = "".join(aa for _, aa in numbering)

                # Get metadata from assignment_details
                metadata = assignment_details[i][domain_idx] if assignment_details else {}
                if isinstance(metadata, dict):
                    chain_type = metadata.get("chain_type", "Unknown")
                    species = metadata.get("species", "Unknown")
                    bitscore = metadata.get("bitscore", "N/A")
                else:
                    chain_type = "Unknown"
                    species = "Unknown"
                    bitscore = "N/A"

                chain_name = {"H": "VH", "K": "VK", "L": "VL"}.get(
                    chain_type, "Unknown"
                )

                # Write to CSV
                writer.writerow(
                    [
                        seq_id,
                        chain_name,
                        species,
                        bitscore,
                        start,
                        end,
                        cdr1,
                        cdr2,
                        cdr3,
                        f'"{aligned_seq}"',  # aligned sequences read as text
                    ]
                )
        else:
            # For rows with no detected domain
            writer.writerow([seq_id, "No_domain", "", "", "", "", "", "", "", ""])

print(f"VH sequences analyzed using ANARCI. Output saved to {output_csv}")

VH sequences analyzed using ANARCI. Output saved to /content/drive/MyDrive/poster_antibodies/anarci_vh.csv


In [6]:
# @title IMGT numbering for VL sequences

# Load CSV file
input_csv = "/content/drive/MyDrive/poster_antibodies/antibody_temp.csv"
output_dir = "/content/drive/MyDrive/poster_antibodies"
output_csv = os.path.join(output_dir, "anarci_vl.csv")  #change output file name as needed

sequences = []
with open(input_csv, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        seq_id = row.get("antibody_id")
        seq = row["vl_protein_sequence"].replace(" ", "").upper()  #remove spaces, ensure uppercase
        sequences.append((seq_id, seq))

# Run ANARCI in batch
results, assignment_details, hit_tables = anarci(sequences, scheme="imgt")

# Open output CSV
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "antibody_id", "chain_type", "species", "bitscore",
        "start", "end", "cdr1", "cdr2", "cdr3", "aligned_imgt"
    ])

    # Process each sequence
    for i, (seq_id, seq) in enumerate(sequences):
        if results[i] and results[i][0]:
            for domain_idx, (numbering, start, end) in enumerate(results[i]):

                def extract_region(numbering, start_pos, end_pos):
                    return "".join(
                        aa for ((pos, _), aa) in numbering
                        if start_pos <= pos <= end_pos and aa != "-"
                    )

                cdr1 = extract_region(numbering, 27, 38)
                cdr2 = extract_region(numbering, 56, 65)
                cdr3 = extract_region(numbering, 105, 117)
                aligned_seq = "".join(aa for _, aa in numbering)

                # Get metadata from assignment_details
                metadata = assignment_details[i][domain_idx] if assignment_details else {}
                if isinstance(metadata, dict):
                    chain_type = metadata.get("chain_type", "Unknown")
                    species = metadata.get("species", "Unknown")
                    bitscore = metadata.get("bitscore", "N/A")
                else:
                    chain_type = "Unknown"
                    species = "Unknown"
                    bitscore = "N/A"

                chain_name = {"H": "VH", "K": "VK", "L": "VL"}.get(chain_type, "Unknown")

                # Write to CSV
                writer.writerow([
                    seq_id,
                    chain_name,
                    species,
                    bitscore,
                    start,
                    end,
                    cdr1,
                    cdr2,
                    cdr3,
                    f'"{aligned_seq}"' # aligned sequences read as text
                ])
        else:
            # For rows with no detected domain
            writer.writerow([seq_id, "No_domain", "", "", "", "", "", "", "", ""])

print(f"VL sequences analyzed using ANARCI. Output saved to {output_csv}")

VL sequences analyzed using ANARCI. Output saved to /content/drive/MyDrive/poster_antibodies/anarci_vl.csv


In [7]:
# @title Generate file with IMGT numbering and melting temperature

# Read CSV file
vh_path = "/content/drive/MyDrive/poster_antibodies/anarci_vh.csv" #VH numberings
vl_path = "/content/drive/MyDrive/poster_antibodies/anarci_vl.csv" #VL numberings
seq_path = "/content/drive/MyDrive/poster_antibodies/antibody_temp.csv"
output_dir = "/content/drive/MyDrive/poster_antibodies"

# Read CSV file
vh_df = pd.read_csv(vh_path)
vl_df = pd.read_csv(vl_path)
seq_df = pd.read_csv(seq_path)


# Extract columns
vhseq_df = seq_df[
    ["antibody_id", "vh_protein_sequence", "temp"]
    ]

vlseq_df = seq_df[
    ["antibody_id", "vl_protein_sequence", "temp"]
    ]

vh_df = vh_df[
    ["antibody_id", "cdr1", "cdr2", "cdr3", "aligned_imgt"]
    ]

vl_df = vl_df[
    ["antibody_id", "cdr1", "cdr2", "cdr3", "aligned_imgt"]]

# ===== Merge VH =====

# Merge dataframes on antibody_id
merged_df = pd.merge(
    vhseq_df,
    vh_df,
    on="antibody_id",
    how="inner"
)

df_clean = merged_df.dropna(subset=["temp"]) #drops antibodies with missing temperature

# Write to CSV
output_filepath_vh = os.path.join(output_dir, "merged_anarci_vh.csv")
df_clean.to_csv(output_filepath_vh, index=False)
print(f"Numbering for VH saved as CSV: {output_filepath_vh}")


# ===== Merge VL =====

# Merge dataframes on antibody_id
df_merged = pd.merge(
    vlseq_df,
    vl_df,
    on="antibody_id",
    how="inner"
)

clean_df = df_merged.dropna(subset=["temp"]) #drops antibodies with missing temperature

# Write to CSV
output_filepath_vl = os.path.join(output_dir, "merged_anarci_vl.csv")
clean_df.to_csv(output_filepath_vl, index=False)
print(f"Numbering for VL saved as CSV: {output_filepath_vl}")

Numbering for VH saved as CSV: /content/drive/MyDrive/poster_antibodies/merged_anarci_vh.csv
Numbering for VL saved as CSV: /content/drive/MyDrive/poster_antibodies/merged_anarci_vl.csv


# ESM-2 Embeddings

Goal: embeddings of aligned sequences

⏳ 12 minutes (CPU) | 3 minutes (CPU)

In [8]:
# @title Create a working directory on Drive
output_dir = "/content/drive/MyDrive/esm_antibody_results"
!mkdir -p {output_dir}
print(f"Results will be saved to: {output_dir}")

Results will be saved to: /content/drive/MyDrive/esm_antibody_results


In [9]:
# @title ESM embeddings VH sequences

# ===== Parameters =====

csv_file = "/content/drive/MyDrive/poster_antibodies/merged_anarci_vh.csv"   #must contain antibody_id and aligned_imgt
prefix = "vh_esm2"
length = 127
processor = torch.device("cuda" if torch.cuda.is_available() else "cpu") #use GPU if available
output_dir = "/content/drive/MyDrive/esm_antibody_results" # Ensure output_dir is defined here

# Load CSV
df = pd.read_csv(csv_file)

# Padding function
def pad_sequence(seq, template_len):
    return seq.ljust(template_len, "-")[:template_len]

# Remove leading and lagging quote marks
df["aligned_imgt_padded"] = df["aligned_imgt"].apply(
    lambda x: pad_sequence(x.strip('"').strip(), length)
)

# ===== ESM-2 Embeddings =====

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()  # 1280-dim
batch_converter = alphabet.get_batch_converter()
model.eval()
model.to(processor)

emb_size = model.embed_dim  # 1280

# Embedding function
def embed_sequence(seq):

    mask = torch.tensor([aa != "-" for aa in seq], device=processor)
    seq_no_gaps = "".join([aa for aa in seq if aa != "-"])

    # If sequence only has gaps return zeros
    if len(seq_no_gaps) == 0:
        per_residue = torch.zeros((length, emb_size))
        per_sequence = torch.zeros((emb_size,))
        return per_residue, per_sequence

    # Convert sequences to tokens
    _, _, tokens = batch_converter([(None, seq_no_gaps)]) #tokens are CLS + seq + EOS
    tokens = tokens.to(processor)
    with torch.no_grad():
        out = model(tokens, repr_layers=[30], return_contacts=False)
    residue_emb = out["representations"][30][0, 1:-1] #tokens now are seq only

    # Map embeddings to padded positions
    per_residue = torch.zeros((length, emb_size), device=processor)
    per_residue[mask] = residue_emb
    per_sequence = per_residue[mask].mean(0)

    # Clean GPU memory
    del tokens, out, residue_emb
    gc.collect()
    return per_residue.cpu(), per_sequence.cpu()


# Run embedding
N = len(df)

X_residue = np.zeros((N, length, emb_size), dtype=np.float32)
X_sequence = np.zeros((N, emb_size), dtype=np.float32)
antibody_ids = []

for i, (ab_id, seq) in enumerate(
        zip(df["antibody_id"], df["aligned_imgt_padded"])
    ):
    print(f"Embedding {i+1}/{N}: {ab_id}")
    per_res, per_seq = embed_sequence(seq)

    X_residue[i] = per_res.numpy()
    X_sequence[i] = per_seq.numpy()
    antibody_ids.append(ab_id)

# ===== Save Outputs =====
np.save(os.path.join(output_dir, f"{prefix}_per_residue.npy"), X_residue) # per_residue: [127, 1280]
np.save(os.path.join(output_dir, f"{prefix}_per_sequence.npy"), X_sequence) # per_sequence: [1280]
np.save(os.path.join(output_dir, f"{prefix}_antibody_ids.npy"), np.array(antibody_ids))
np.save(os.path.join(output_dir, f"{prefix}_imgt_positions.npy"),
        np.arange(1, length + 1))

print("Embeddings for VH domain saved successfully")

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Embedding 1/208: GDPa1-001
Embedding 2/208: GDPa1-002
Embedding 3/208: GDPa1-003
Embedding 4/208: GDPa1-004
Embedding 5/208: GDPa1-005
Embedding 6/208: GDPa1-006
Embedding 7/208: GDPa1-007
Embedding 8/208: GDPa1-008
Embedding 9/208: GDPa1-010
Embedding 10/208: GDPa1-011
Embedding 11/208: GDPa1-012
Embedding 12/208: GDPa1-014
Embedding 13/208: GDPa1-015
Embedding 14/208: GDPa1-016
Embedding 15/208: GDPa1-017
Embedding 16/208: GDPa1-018
Embedding 17/208: GDPa1-019
Embedding 18/208: GDPa1-020
Embedding 19/208: GDPa1-021
Embedding 20/208: GDPa1-022
Embedding 21/208: GDPa1-023
Embedding 22/208: GDPa1-024
Embedding 23/208: GDPa1-027
Embedding 24/208: GDPa1-028
Emb

In [10]:
# @title ESM embeddings VL sequences

# ===== Parameters =====

csv_file = "/content/drive/MyDrive/poster_antibodies/merged_anarci_vl.csv"   #must contain antibody_id and aligned_imgt
prefix = "vl_esm2"
length = 127
processor = torch.device("cuda" if torch.cuda.is_available() else "cpu") #use GPU if available
output_dir = "/content/drive/MyDrive/esm_antibody_results" # Define output_dir for saving

# Load CSV
df = pd.read_csv(csv_file)

# Padding function
def pad_sequence(seq, template_len):
    return seq.ljust(template_len, "-")[:template_len]

# Remove leading and lagging quote marks
df["aligned_imgt_padded"] = df["aligned_imgt"].apply(
    lambda x: pad_sequence(x.strip('"').strip(), length)
)

# ===== ESM-2 Embeddings =====

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()  # 1280-dim
batch_converter = alphabet.get_batch_converter()
model.eval()
model.to(processor)

emb_size = model.embed_dim  # 1280

# Embedding function
def embed_sequence(seq):

    mask = torch.tensor([aa != "-" for aa in seq], device=processor)
    seq_no_gaps = "".join([aa for aa in seq if aa != "-"])

    # If sequence only has gaps return zeros
    if len(seq_no_gaps) == 0:
        per_residue = torch.zeros((length, emb_size))
        per_sequence = torch.zeros((emb_size,))
        return per_residue, per_sequence

    # Convert sequences to tokens
    _, _, tokens = batch_converter([(None, seq_no_gaps)]) #tokens are CLS + seq + EOS
    tokens = tokens.to(processor)
    with torch.no_grad():
        out = model(tokens, repr_layers=[30], return_contacts=False)
    residue_emb = out["representations"][30][0, 1:-1] #tokens now are seq only

    # Map embeddings to padded positions
    per_residue = torch.zeros((length, emb_size), device=processor)
    per_residue[mask] = residue_emb
    per_sequence = per_residue[mask].mean(0)

    # Clean GPU memory
    del tokens, out, residue_emb
    gc.collect()
    return per_residue.cpu(), per_sequence.cpu()


# Run embedding
N = len(df)

X_residue = np.zeros((N, length, emb_size), dtype=np.float32)
X_sequence = np.zeros((N, emb_size), dtype=np.float32)
antibody_ids = []

for i, (ab_id, seq) in enumerate(
        zip(df["antibody_id"], df["aligned_imgt_padded"])
    ):
    print(f"Embedding {i+1}/{N}: {ab_id}")
    per_res, per_seq = embed_sequence(seq)

    X_residue[i] = per_res.numpy()
    X_sequence[i] = per_seq.numpy()
    antibody_ids.append(ab_id)

# ===== Save Outputs =====
np.save(os.path.join(output_dir, f"{prefix}_per_residue.npy"), X_residue) # per_residue: [127, 1280]
np.save(os.path.join(output_dir, f"{prefix}_per_sequence.npy"), X_sequence) # per_sequence: [1280]
np.save(os.path.join(output_dir, f"{prefix}_antibody_ids.npy"), np.array(antibody_ids))
np.save(os.path.join(output_dir, f"{prefix}_imgt_positions.npy"),
        np.arange(1, length + 1))

print("Embeddings for VL domain saved successfully")

Embedding 1/208: GDPa1-001
Embedding 2/208: GDPa1-002
Embedding 3/208: GDPa1-003
Embedding 4/208: GDPa1-004
Embedding 5/208: GDPa1-005
Embedding 6/208: GDPa1-006
Embedding 7/208: GDPa1-007
Embedding 8/208: GDPa1-008
Embedding 9/208: GDPa1-010
Embedding 10/208: GDPa1-011
Embedding 11/208: GDPa1-012
Embedding 12/208: GDPa1-014
Embedding 13/208: GDPa1-015
Embedding 14/208: GDPa1-016
Embedding 15/208: GDPa1-017
Embedding 16/208: GDPa1-018
Embedding 17/208: GDPa1-019
Embedding 18/208: GDPa1-020
Embedding 19/208: GDPa1-021
Embedding 20/208: GDPa1-022
Embedding 21/208: GDPa1-023
Embedding 22/208: GDPa1-024
Embedding 23/208: GDPa1-027
Embedding 24/208: GDPa1-028
Embedding 25/208: GDPa1-029
Embedding 26/208: GDPa1-030
Embedding 27/208: GDPa1-031
Embedding 28/208: GDPa1-032
Embedding 29/208: GDPa1-033
Embedding 30/208: GDPa1-034
Embedding 31/208: GDPa1-035
Embedding 32/208: GDPa1-036
Embedding 33/208: GDPa1-037
Embedding 34/208: GDPa1-038
Embedding 35/208: GDPa1-040
Embedding 36/208: GDPa1-042
E

In [11]:
# @title Confirm shape of embeddings

# ===== Per-residue =====

# Load embeddings
X_vh = np.load("/content/drive/MyDrive/esm_antibody_results/vh_esm2_per_residue.npy")
X_vl = np.load("/content/drive/MyDrive/esm_antibody_results/vl_esm2_per_residue.npy")

# Display shape of embeddings
print("VH (per-residue) shape:", X_vh.shape)
print("VL (per-residue) shape:", X_vl.shape)

# ===== Per-sequence =====

# Load embeddings
X_vh = np.load("/content/drive/MyDrive/esm_antibody_results/vh_esm2_per_sequence.npy")
X_vl = np.load("/content/drive/MyDrive/esm_antibody_results/vl_esm2_per_sequence.npy")

# Display shape of embeddings
print("VH (per-sequence) shape:", X_vh.shape)
print("VL (per-sequence) shape:", X_vl.shape)

VH (per-residue) shape: (208, 127, 1280)
VL (per-residue) shape: (208, 127, 1280)
VH (per-sequence) shape: (208, 1280)
VL (per-sequence) shape: (208, 1280)


In [12]:
# @title Merge ESM embeddings


# ===== Load embeddings =====

# Define the output directory for loading and saving
output_dir = "/content/drive/MyDrive/esm_antibody_results"

# Load embeddings
X_vh = np.load(os.path.join(output_dir, "vh_esm2_per_residue.npy"))
Xseq_vh = np.load(os.path.join(output_dir, "vh_esm2_per_sequence.npy"))
ids_vh = np.load(os.path.join(output_dir, "vh_esm2_antibody_ids.npy"))

X_vl = np.load(os.path.join(output_dir, "vl_esm2_per_residue.npy"))
Xseq_vl = np.load(os.path.join(output_dir, "vl_esm2_per_sequence.npy"))
ids_vl = np.load(os.path.join(output_dir, "vl_esm2_antibody_ids.npy"))

# Build index maps
vh_map = {ab_id: i for i, ab_id in enumerate(ids_vh)}
vl_map = {ab_id: i for i, ab_id in enumerate(ids_vl)}


# ===== Pair and merge embeddings =====

# Pair antibodies
paired_ids = sorted(set(ids_vh).intersection(ids_vl))
print(f"Paired antibodies: {len(paired_ids)}")


# Merge per-residue embeddings
N = len(paired_ids)
emb_size = X_vh.shape[2]

X_paired = np.zeros((N, 254, emb_size), dtype=np.float32)

for i, ab_id in enumerate(paired_ids):
    X_paired[i] = np.concatenate(
        [X_vh[vh_map[ab_id]], X_vl[vl_map[ab_id]]],
        axis=0
    )

# Merge per-sequence embeddings
embed_size = Xseq_vh.shape[1]

Xseq_paired = np.zeros((N, embed_size * 2), dtype=np.float32)

for i, ab_id in enumerate(paired_ids):
    Xseq_paired[i] = np.concatenate(
        [Xseq_vh[vh_map[ab_id]], Xseq_vl[vl_map[ab_id]]],
        axis=0
    )


# ===== Save embeddings =====

# Save per-residue embeddings
np.save(os.path.join(output_dir, "paired_vh_vl_per_residue.npy"), X_paired)
np.save(os.path.join(output_dir, "paired_antibody_ids.npy"), np.array(paired_ids))

# Save per-sequence embeddings
np.save(os.path.join(output_dir, "paired_vh_vl_per_sequence.npy"), Xseq_paired)

# Confirm and display pairing
print("Paired per-residue shape:", X_paired.shape)
print("Paired per-sequence shape:", Xseq_paired.shape)

Paired antibodies: 208
Paired per-residue shape: (208, 254, 1280)
Paired per-sequence shape: (208, 2560)
